---
# Copyright (c) 2026 Angela Villota, and collaborators from the CyED block
# Licensed under the PolyForm Noncommercial License 1.0.0.
# Commercial use is prohibited without prior written authorization.
title: Regular-expression evaluation preparation
---

[Open this notebook in Google Colab](https://colab.research.google.com/github/angievig/CyED3/blob/main/Week2/SeguimientosAnteriores/regex-evaluation-prep.ipynb)

This notebook reorganizes themes from previous activities into guided practice. It does not provide a complete answer key. Each task includes tests and a partial construction so you can diagnose your own work.

## Suggested routine

1. Translate every requirement into a small pattern obligation.
2. Choose the Python operation (`fullmatch`, `search`, `finditer`, or `sub`).
3. Write positive, negative, and boundary tests.
4. Combine the obligations.
5. Explain every part of the final pattern in words.

In [ ]:
import re
from collections import Counter

TODO_PATTERN = r"(?!)"  # A safe placeholder that deliberately matches nothing.

## Testing helpers

These helpers report results without stopping the notebook. A failing case is evidence about what to revise.

In [ ]:
def check_fullmatch(pattern, cases, flags=0):
    passed = 0
    for value, expected in cases:
        actual = re.fullmatch(pattern, value, flags=flags) is not None
        mark = "✓" if actual == expected else "✗"
        print(f"{mark} {value!r:30} expected={expected} actual={actual}")
        passed += actual == expected
    print(f"Passed {passed}/{len(cases)} cases")
    return passed == len(cases)


def check_function(function, cases):
    passed = 0
    for value, expected in cases:
        actual = function(value)
        mark = "✓" if actual == expected else "✗"
        print(f"{mark} input={value!r}\n    expected={expected!r}\n    actual  ={actual!r}")
        passed += actual == expected
    print(f"Passed {passed}/{len(cases)} cases")
    return passed == len(cases)

# Part A — Formal regular expressions

Solve these on paper before writing Python. Use $|$ for union and juxtaposition for concatenation.

## Exercise 1 — Exactly one occurrence of `aaa`

Over $\Sigma=\{a,b,c\}$, design a regular expression for strings containing exactly one occurrence of three consecutive `a` symbols. Overlaps count: `aaaa` contains two occurrences of `aaa`.

<details><summary>Strategy</summary>

A naive expression $\Sigma^*aaa\Sigma^*$ guarantees at least one occurrence, not exactly one. Build a language that avoids `aaa`, then place the required `aaa` between a restricted prefix and suffix. The prefix cannot end in `a`, and the suffix cannot begin with `a`, or an overlapping occurrence will be created.

</details>

## Exercise 2 — Even length

Over $\Sigma=\{0,1,2\}$, design a regular expression for all strings whose length is divisible by two.

<details><summary>Partial construction</summary>

Let $S=(0|1|2)$. A block of two arbitrary symbols is $SS$. Repeat that block zero or more times. Check whether the empty string should be included.

</details>

# Part B — Complete-string validation

## Exercise 3 — Mirrored username

Validate usernames satisfying all requirements:

- length from 8 to 16 characters;
- first character is a letter;
- at least one underscore;
- at least one digit;
- final character equals the first, case-sensitively; and
- only letters, digits, and underscores.

In [ ]:
username_cases = [
    ("User_123U", True),
    ("User_1234", False),
    ("a_user1a", True),
    ("Z_user_9Z", True),
    ("A123456_7A", True),
    ("a__1b", False),
    ("_User_1_", False),
    ("abc_def_1c", False),
]

username_pattern = TODO_PATTERN  # TODO
check_fullmatch(username_pattern, username_cases)

<details><summary>Partial construction</summary>

Capture the initial letter with `([A-Za-z])`. Use one lookahead to require a digit and another to require an underscore. Finish with a backreference to the first group. Because the first and final letters consume two characters, the middle repetition must account for the remaining 6–14 characters.

</details>

## Exercise 4 — Mirrored time

Validate `HH:MM:MM:HH` such that both hours are equal, both minutes are equal, hours range from `00` to `23`, and minutes range from `00` to `59`.

In [ ]:
mirrored_time_cases = [
    ("12:30:30:12", True),
    ("00:00:00:00", True),
    ("23:59:59:23", True),
    ("12:30:31:12", False),
    ("12:60:60:12", False),
    ("24:00:00:24", False),
    ("12:30:30:13", False),
]

mirrored_time_pattern = TODO_PATTERN  # TODO
check_fullmatch(mirrored_time_pattern, mirrored_time_cases)

<details><summary>Partial construction</summary>

First capture a valid hour: `([01][0-9]|2[0-3])`. Then capture a valid minute. The second minute and hour should be backreferences rather than new range patterns.

</details>

## Exercise 5 — Simplified ISO timestamp

Validate `YYYY-MM-DDTHH:MM:SS`. Capture the year, month, and day separately. Restrict month to `01`–`12`, day to `01`–`31`, hour to `00`–`23`, and minute/second to `00`–`59`.

This is format validation; it does not need to reject calendar impossibilities such as June 31.

In [ ]:
iso_cases = [
    ("2023-05-15T14:30:05", ("2023", "05", "15")),
    ("1999-12-31T23:59:59", ("1999", "12", "31")),
    ("2023-13-01T14:30:00", None),
    ("2023-01-32T14:30:00", None),
    ("2023-5-15T14:30:00", None),
    ("2023-05-15 14:30:00", None),
]

iso_pattern = re.compile(TODO_PATTERN)  # TODO

def parse_iso_timestamp(value):
    match = iso_pattern.fullmatch(value)
    return match.groups() if match else None

check_function(parse_iso_timestamp, iso_cases)

<details><summary>Partial construction</summary>

Build independent fragments before joining them:

```python
year = r"(\d{4})"
month = r"(0[1-9]|1[0-2])"
day = r"(0[1-9]|[12]\d|3[01])"
hour = r"(?:[01]\d|2[0-3])"
minute_or_second = r"(?:[0-5]\d)"
```

Only year, month, and day should be capturing groups if `groups()` must return exactly three values.

</details>

## Exercise 6 — Product SKU

Validate a SKU with two uppercase department letters, a hyphen, four year digits, another hyphen, and a serial containing at least three alphanumeric characters whose final character is a digit.

In [ ]:
sku_cases = [
    ("EL-2023-A109", True),
    ("TS-1998-999ZZ0", True),
    ("el-2023-A109", False),
    ("E1-2023-A109", False),
    ("EL-23-A109", False),
    ("EL-2023-ABC", False),
    ("EL-2023-A1", False),
    ("EL2023-A109", False),
]

sku_pattern = TODO_PATTERN  # TODO
check_fullmatch(sku_pattern, sku_cases)

<details><summary>Hint</summary>

Translate each requirement in order. If the entire serial has at least three characters and its final character is handled separately, how many alphanumeric characters must appear before that final digit?

</details>

# Part C — Extraction and text analysis

## Exercise 7 — Monetary values

Extract USD (`$`) and EUR (`€`) amounts. The value must have exactly two decimal places. Thousands separators are optional, but if commas appear they must divide groups of three digits. Return `(currency, value)` tuples.

In [ ]:
money_cases = [
    ("$1,250.00", [("$", "1,250.00")]),
    ("€45.50", [("€", "45.50")]),
    ("$5.00 and €10.99", [("$", "5.00"), ("€", "10.99")]),
    ("$1,000,000.00", [("$", "1,000,000.00")]),
    ("$100", []),
    ("€45.5", []),
    ("$1,20.00", []),
]

money_pattern = re.compile(TODO_PATTERN)  # TODO

def extract_money(text):
    return money_pattern.findall(text)

check_function(extract_money, money_cases)

<details><summary>Partial construction</summary>

Capture the symbol with `([$€])`. The integer part has two alternatives: comma-grouped digits or ungrouped digits. Use a noncapturing group for that choice so the result contains only the requested two captures.

</details>

## Exercise 8 — Mentions and hashtags

Extract mentions and hashtags. Mentions begin with `@` and contain letters, digits, or underscores. Hashtags begin with `#`, use the same character set, and must contain at least one letter so `#123` is rejected. Do not extract `@domain` from an email.

In [ ]:
social_cases = [
    ("#Python", ["#Python"]),
    ("@Guido", ["@Guido"]),
    ("#12345", []),
    ("#Data_Science and @user123", ["#Data_Science", "@user123"]),
    ("email@domain.com", []),
    ("#Regex2026", ["#Regex2026"]),
]

social_pattern = re.compile(TODO_PATTERN)  # TODO

def get_social_tags(text):
    return social_pattern.findall(text)

check_function(get_social_tags, social_cases)

<details><summary>Strategy</summary>

Use separate alternatives for mentions and hashtags, or factor their common structure carefully. A negative lookbehind can prevent a match immediately after a word character or period. A lookahead after `#` can require a letter before the tag ends.

</details>

## Exercise 9 — Repeated word triplets

Find every three-word sequence that occurs at least twice, case-insensitively. Punctuation and repeated spaces should not prevent matching. Preserve the order in which repeated triplets first appear.

In [ ]:
triplet_cases = [
    ("the quick brown fox jumps over the quick brown fox",
     ["the quick brown", "quick brown fox"]),
    ("Hello world again. hello WORLD again.",
     ["hello world again"]),
    ("every word is different in this sentence", []),
    ("one two three two three four one two three", ["one two three"]),
]

word_pattern = re.compile(r"[A-Za-z]+(?:\'[A-Za-z]+)?")

def repeated_triplets(text):
    words = [match.group().lower() for match in word_pattern.finditer(text)]
    # TODO: build consecutive 3-tuples, count them, and preserve first appearance.
    return []

check_function(repeated_triplets, triplet_cases)

<details><summary>Partial construction</summary>

The regex already extracts normalized word tokens. Build triplets with `zip(words, words[1:], words[2:])`, count them with `Counter`, and join those whose count is at least two. Use a `seen` set if needed to avoid returning the same repeated triplet twice.

</details>

## Exercise 10 — SQL aliases

Extract `(table, alias)` from `FROM table AS alias` and subsequent `JOIN table AS alias` clauses. Names contain letters, digits, or underscores. Reject a clause when the table and alias are the same, case-insensitively. A query beginning only with `JOIN` is invalid for this task.

In [ ]:
sql_cases = [
    ("FROM users AS u", [("users", "u")]),
    ("FROM Orders AS o", [("Orders", "o")]),
    ("FROM logs AS logs", []),
    ("FROM products AS p JOIN sales AS s", [("products", "p"), ("sales", "s")]),
    ("SELECT * FROM users", []),
    ("join loans AS l", []),
]

alias_pattern = re.compile(TODO_PATTERN, re.IGNORECASE)  # TODO

def find_table_aliases(query):
    if not re.search(r"\bfrom\b", query, re.IGNORECASE):
        return []
    return alias_pattern.findall(query)

check_function(find_table_aliases, sql_cases)

<details><summary>Partial construction</summary>

Start with `\b(?:from|join)\s+` and capture the table. After `\s+as\s+`, use a negative lookahead with a backreference to reject the same complete name before capturing the alias. Remember that `re.IGNORECASE` also affects the backreference comparison.

</details>

# Final preparation task

Choose two exercises—one validation problem and one extraction problem—and add at least five new tests to each. For every new negative test, state which single requirement it violates.

Then write a short explanation using this structure:

1. **Operation:** why `fullmatch`, `findall`, or `finditer` is appropriate.
2. **Structure:** what each group or alternative represents.
3. **Restrictions:** how ranges, lookarounds, or boundaries enforce requirements.
4. **Evidence:** which boundary tests give confidence in the result.

In [ ]:
# Add your own tests here.
additional_validation_cases = [
    # ("value", True),
]

additional_extraction_cases = [
    # ("text", ["expected", "matches"]),
]

## Readiness checklist

- [ ] I can translate each written requirement into one pattern responsibility.
- [ ] I know when the entire string must match.
- [ ] I can distinguish capturing and noncapturing groups.
- [ ] I can use a backreference to enforce repeated text.
- [ ] I can use lookarounds for requirements that should not consume text.
- [ ] I test minimum, maximum, empty, malformed, and near-miss cases.
- [ ] I can explain my expression without reading it character by character.